<a href="https://colab.research.google.com/github/Oruntu-Tanima-Proje/otProje/blob/main/notebooks/06_karsilastirma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 06 - Model Karşılaştırma Raporu

## 3 CNN Modelinin Detaylı Analizi

Bu notebook, eğitilmiş 3 modeli (**MobileNetV2**, **ResNet50**, **EfficientNetB0**)
çeşitli metrikler ve görseller kullanarak karşılaştırır.

### Karşılaştırma Yöntemleri
1. ✅ Genel metrik tablosu (accuracy, precision, recall, F1)
2. ✅ Test accuracy bar grafiği
3. ✅ F1-score karşılaştırması
4. ✅ Model boyutu analizi
5. ✅ Trade-off grafiği (accuracy vs boyut)
6. ✅ Confusion matrix (3 model)
7. ✅ Sınıf bazlı F1 karşılaştırması

### Beklenen Sonuçlar
- En yüksek doğruluk: **ResNet50** (~%98.65)
- En iyi denge: **EfficientNetB0** (~%96.55)
- En hafif: **MobileNetV2** (~%92.40)

In [ ]:
# ============================================================
# 1. HAZIRLIK - Modelleri ve veriyi Drive'dan al
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficient_preprocess

IMG_SIZE = 224
BATCH_SIZE = 32
drive_proje = "/content/drive/MyDrive/Domates_Projesi"

# Veriyi Drive'dan kopyala
if not os.path.exists("tomato_data"):
    print("📦 Veri seti Drive'dan kopyalanıyor...")
    shutil.copytree(f"{drive_proje}/data", "tomato_data")
    print("   ✅ Tamamlandı")

# Modelleri Drive'dan kopyala
os.makedirs("models", exist_ok=True)
print("\n🧠 Modeller kopyalanıyor...")
for model_file in os.listdir(f"{drive_proje}/models"):
    src = f"{drive_proje}/models/{model_file}"
    dst = f"models/{model_file}"
    if not os.path.exists(dst):
        shutil.copy(src, dst)

# 3 modeli yükle
print("\n📥 Modeller yükleniyor...")
mobilenet_model = load_model('models/mobilenetv2_final.keras')
resnet_model = load_model('models/resnet50_final.keras')
eff_model = load_model('models/efficientnetb0_final.keras')
print("   ✅ MobileNetV2 yüklendi")
print("   ✅ ResNet50 yüklendi")
print("   ✅ EfficientNetB0 yüklendi")

In [ ]:
# ============================================================
# 2. HER MODEL İÇİN ÖZEL TEST GENERATOR'LARI
# (Her modelin preprocessing'i farklı!)
# ============================================================

# MobileNetV2 için (rescale=1./255)
test_datagen_mobile = ImageDataGenerator(rescale=1./255)
test_generator_mobile = test_datagen_mobile.flow_from_directory(
    "tomato_data/test", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

# ResNet50 için (caffe-style preprocessing)
test_datagen_resnet = ImageDataGenerator(preprocessing_function=resnet_preprocess)
test_generator_resnet = test_datagen_resnet.flow_from_directory(
    "tomato_data/test", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

# EfficientNetB0 için (kendi preprocess'i)
test_datagen_eff = ImageDataGenerator(preprocessing_function=efficient_preprocess)
test_generator_eff = test_datagen_eff.flow_from_directory(
    "tomato_data/test", target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

print("\n✅ 3 model için test generator'ları hazır")

In [ ]:
# ============================================================
# 3. 3 MODELİN TEST PERFORMANSINI HESAPLA
# ============================================================

from sklearn.metrics import precision_score, recall_score, f1_score

print("=" * 70)
print("🧪 3 MODEL TEST DEĞERLENDİRMESİ")
print("=" * 70)

# MobileNetV2
print("\n🔵 MobileNetV2 değerlendiriliyor...")
test_generator_mobile.reset()
loss_mobile, acc_mobile = mobilenet_model.evaluate(test_generator_mobile, verbose=0)
test_generator_mobile.reset()
pred_mobile = mobilenet_model.predict(test_generator_mobile, verbose=0)
y_pred_mobile = np.argmax(pred_mobile, axis=1)
y_true = test_generator_mobile.classes
prec_mobile = precision_score(y_true, y_pred_mobile, average='weighted')
rec_mobile = recall_score(y_true, y_pred_mobile, average='weighted')
f1_mobile = f1_score(y_true, y_pred_mobile, average='weighted')
size_mobile = os.path.getsize('models/mobilenetv2_final.keras') / (1024 * 1024)
print(f"   Test Acc: {acc_mobile*100:.2f}% | F1: {f1_mobile:.4f}")

# ResNet50
print("\n🔴 ResNet50 değerlendiriliyor...")
test_generator_resnet.reset()
loss_resnet, acc_resnet = resnet_model.evaluate(test_generator_resnet, verbose=0)
test_generator_resnet.reset()
pred_resnet = resnet_model.predict(test_generator_resnet, verbose=0)
y_pred_resnet = np.argmax(pred_resnet, axis=1)
prec_resnet = precision_score(y_true, y_pred_resnet, average='weighted')
rec_resnet = recall_score(y_true, y_pred_resnet, average='weighted')
f1_resnet = f1_score(y_true, y_pred_resnet, average='weighted')
size_resnet = os.path.getsize('models/resnet50_final.keras') / (1024 * 1024)
print(f"   Test Acc: {acc_resnet*100:.2f}% | F1: {f1_resnet:.4f}")

# EfficientNetB0
print("\n🟢 EfficientNetB0 değerlendiriliyor...")
test_generator_eff.reset()
loss_eff, acc_eff = eff_model.evaluate(test_generator_eff, verbose=0)
test_generator_eff.reset()
pred_eff = eff_model.predict(test_generator_eff, verbose=0)
y_pred_eff = np.argmax(pred_eff, axis=1)
prec_eff = precision_score(y_true, y_pred_eff, average='weighted')
rec_eff = recall_score(y_true, y_pred_eff, average='weighted')
f1_eff = f1_score(y_true, y_pred_eff, average='weighted')
size_eff = os.path.getsize('models/efficientnetb0_final.keras') / (1024 * 1024)
print(f"   Test Acc: {acc_eff*100:.2f}% | F1: {f1_eff:.4f}")

print("\n✅ 3 model değerlendirmesi tamamlandı")

In [ ]:
# ============================================================
# 4. KARŞILAŞTIRMA TABLOSU
# ============================================================

import pandas as pd

results_data = {
    'Model': ['MobileNetV2', 'ResNet50', 'EfficientNetB0'],
    'Test Accuracy (%)': [acc_mobile*100, acc_resnet*100, acc_eff*100],
    'Precision': [prec_mobile, prec_resnet, prec_eff],
    'Recall': [rec_mobile, rec_resnet, rec_eff],
    'F1-Score': [f1_mobile, f1_resnet, f1_eff],
    'Test Loss': [loss_mobile, loss_resnet, loss_eff],
    'Model Boyutu (MB)': [size_mobile, size_resnet, size_eff],
    'Parametre (M)': [2.4, 24.0, 5.3]
}

df_results = pd.DataFrame(results_data)
print("=" * 100)
print("📊 3 MODEL KARŞILAŞTIRMA TABLOSU")
print("=" * 100)
print(df_results.to_string(index=False))
print("=" * 100)

# CSV olarak kaydet
df_results.to_csv('karsilastirma_tablosu.csv', index=False)
print("\n✅ Tablo CSV olarak kaydedildi")

In [ ]:
# ============================================================
# 5. KARŞILAŞTIRMA BAR GRAFİKLERİ
# ============================================================

import matplotlib.pyplot as plt

models = ['MobileNetV2', 'ResNet50', 'EfficientNetB0']
test_accs = [acc_mobile*100, acc_resnet*100, acc_eff*100]
f1_scores = [f1_mobile, f1_resnet, f1_eff]
sizes = [size_mobile, size_resnet, size_eff]
colors = ['#3498db', '#e74c3c', '#2ecc71']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Test Accuracy
bars1 = axes[0].bar(models, test_accs, color=colors, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Test Accuracy (%)', fontsize=12, fontweight='bold')
axes[0].set_title('Test Accuracy Karşılaştırması', fontsize=13, fontweight='bold')
axes[0].set_ylim(85, 100)
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars1, test_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.2f}%', ha='center', fontsize=11, fontweight='bold')

# 2. F1-Score
bars2 = axes[1].bar(models, f1_scores, color=colors, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('F1-Score', fontsize=12, fontweight='bold')
axes[1].set_title('F1-Score Karşılaştırması', fontsize=13, fontweight='bold')
axes[1].set_ylim(0.85, 1.0)
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(bars2, f1_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{val:.4f}', ha='center', fontsize=11, fontweight='bold')

# 3. Model Boyutu
bars3 = axes[2].bar(models, sizes, color=colors, edgecolor='black', linewidth=1.5)
axes[2].set_ylabel('Model Boyutu (MB)', fontsize=12, fontweight='bold')
axes[2].set_title('Model Boyutu Karşılaştırması', fontsize=13, fontweight='bold')
axes[2].grid(axis='y', alpha=0.3)
for bar, val in zip(bars3, sizes):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f'{val:.1f} MB', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('🏆 Model Performans Karşılaştırması', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('metrik_karsilastirma.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Bar grafikleri kaydedildi")

In [ ]:
# ============================================================
# 6. TRADE-OFF GRAFİĞİ (Accuracy vs Boyut)
# ============================================================

fig, ax = plt.subplots(figsize=(12, 8))

for i, (model, size, acc) in enumerate(zip(models, sizes, test_accs)):
    ax.scatter(size, acc, s=400, c=colors[i], edgecolors='black', linewidths=2,
               label=model, alpha=0.8, zorder=3)
    ax.annotate(f'{model}\n({acc:.2f}%, {size:.1f}MB)',
                xy=(size, acc), xytext=(15, 10),
                textcoords='offset points', fontsize=11, fontweight='bold')

ax.axhspan(95, 100, alpha=0.1, color='green', label='İdeal Bölge (Yüksek Acc)')
ax.axvspan(0, 50, alpha=0.1, color='blue')

ax.set_xlabel('Model Boyutu (MB)', fontsize=13, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontsize=13, fontweight='bold')
ax.set_title('🎯 Trade-off Analizi: Accuracy vs Model Boyutu\n(Sol üst köşe = ideal)',
             fontsize=14, fontweight='bold')
ax.set_xlim(0, max(sizes) * 1.2)
ax.set_ylim(88, 100)
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig('tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Trade-off grafiği kaydedildi")

In [ ]:
# ============================================================
# 7. 3 MODEL CONFUSION MATRIX
# ============================================================

from sklearn.metrics import confusion_matrix
import seaborn as sns

# Türkçe sınıf isimleri (kısa)
class_names_short = [
    "Bakteriyel Leke", "Erken Yan.", "Geç Yan.", "Yaprak Küfü",
    "Septorya", "Kırmızı Örümcek", "Hedef Leke", "Sarı Yap. Kıv.",
    "Mozaik Virüsü", "Sağlıklı"
]

fig, axes = plt.subplots(1, 3, figsize=(24, 7))

cm_data = [
    (y_true, y_pred_mobile, 'MobileNetV2', 'Blues'),
    (y_true, y_pred_resnet, 'ResNet50', 'Reds'),
    (y_true, y_pred_eff, 'EfficientNetB0', 'Greens')
]

for ax, (yt, yp, model_name, cmap) in zip(axes, cm_data):
    cm = confusion_matrix(yt, yp)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, cbar=False,
                xticklabels=class_names_short, yticklabels=class_names_short, ax=ax,
                annot_kws={'size': 9})
    ax.set_title(f'{model_name}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Tahmin', fontsize=11)
    ax.set_ylabel('Gerçek', fontsize=11)
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.suptitle('🔥 3 Model Confusion Matrix Karşılaştırması (Test Seti)',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrix_all.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Confusion matrix kaydedildi")

In [ ]:
# ============================================================
# 8. SINIF BAZLI F1-SCORE KARŞILAŞTIRMA
# ============================================================

# Her model için sınıf bazlı F1
f1_mobile_class = f1_score(y_true, y_pred_mobile, average=None)
f1_resnet_class = f1_score(y_true, y_pred_resnet, average=None)
f1_eff_class = f1_score(y_true, y_pred_eff, average=None)

x = np.arange(len(class_names_short))
width = 0.27

fig, ax = plt.subplots(figsize=(16, 7))

bars1 = ax.bar(x - width, f1_mobile_class, width, label='MobileNetV2', color='#3498db', edgecolor='black')
bars2 = ax.bar(x, f1_resnet_class, width, label='ResNet50', color='#e74c3c', edgecolor='black')
bars3 = ax.bar(x + width, f1_eff_class, width, label='EfficientNetB0', color='#2ecc71', edgecolor='black')

ax.set_ylabel('F1-Score', fontsize=12, fontweight='bold')
ax.set_xlabel('Hastalık Sınıfı', fontsize=12, fontweight='bold')
ax.set_title('🎯 Sınıf Bazlı F1-Score Karşılaştırması', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(class_names_short, rotation=45, ha='right')
ax.legend(loc='lower right', fontsize=11)
ax.set_ylim(0.7, 1.05)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

# Tablo
print("\n📋 SINIF BAZLI F1-SCORE TABLOSU:")
print("=" * 80)
print(f"{'Sınıf':<25} {'MobileNetV2':>12} {'ResNet50':>12} {'EfficientNetB0':>14}")
print("-" * 80)
for cls, m, r, e in zip(class_names_short, f1_mobile_class, f1_resnet_class, f1_eff_class):
    print(f"{cls:<25} {m:>12.4f} {r:>12.4f} {e:>14.4f}")
print("=" * 80)

## ✅ Model Karşılaştırma Tamamlandı

### 🏆 Final Sonuçlar

| Model | Test Acc | F1-Score | Boyut | Verim |
|-------|----------|----------|-------|-------|
| MobileNetV2 | 92.40% | 0.9242 | 22.74 MB | En hafif |
| **ResNet50** | **98.65%** | **0.9865** | 203.90 MB | En doğru ⭐ |
| EfficientNetB0 | 96.55% | 0.9657 | 29.58 MB | En dengeli ⭐ |

### 🎯 Öneriler

**Hangi model hangi senaryoda kullanılmalı?**

1. **Maksimum doğruluk gereken yerlerde** → **ResNet50**
   - Laboratuvar analizleri
   - Detaylı tıbbi/tarımsal teşhis
   
2. **Doğruluk-hız dengesi gereken yerlerde** → **EfficientNetB0** ⭐
   - Web uygulamaları
   - Bulut tabanlı sistemler
   - Çoğu profesyonel kullanım
   
3. **Mobil/edge cihazlarda** → **MobileNetV2**
   - Telefon uygulamaları
   - IoT cihazları
   - Bağlantısız çalışma

### 📊 Çıktılar
- `karsilastirma_tablosu.csv` — Detaylı metrik tablosu
- `metrik_karsilastirma.png` — Bar grafikleri
- `tradeoff.png` — Trade-off analizi
- `confusion_matrix_all.png` — 3 modelin confusion matrix'i
- `class_f1.png` — Sınıf bazlı F1 karşılaştırması

### Sıradaki Adım
👉 `07_gradcam_analizi.ipynb` — Modellerin yaprağa nereye baktığını görselleştirin (Grad-CAM)